# Notebook 7 : analyse critique

On rassemble ici les expériences qui permettent de répondre à l'énoncé : **discuter la précision** des vitesses déduites de φ, **l'influence du schéma** et du pas de temps sur les résultats dans le labyrinthe, et **interpréter** ce qu'on observe.

**Fichiers nécessaires** : `U.npy`, `V.npy`, `out_domain.txt`, `advection.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('figures', exist_ok=True)
from advection import *

U, V, dom = charger()
fluid, inlet, outlet = preparer_labyrinthe(dom)
c0 = np.zeros(dom.shape)

def lancer(schema='upwind', lim='minmod', temps='euler', cfl=0.5, **kw):
    return simuler(c0, U, V, t_end=20000.0, cfl=cfl, schema=schema, lim=lim, temps=temps,
                   inlet=inlet, outlet=outlet, c_in=1.0, fluid=fluid, **kw)

## 1. Précision des vitesses : où la divergence résiduelle est-elle grande ?

Les vitesses viennent de φ, elle-même solution approchée de Laplace. La divergence ne vaut donc pas exactement 0. Toute divergence non nulle agit comme une **source ou un puits parasite de c**.

In [ ]:
div = (U[:, 1:] - U[:, :-1]) + (V[1:, :] - V[:-1, :])
adiv = np.where(dom == 1, np.abs(div), np.nan)

print("Divergence max       :", np.nanmax(adiv))
print("Divergence moyenne   :", np.nanmean(adiv))
print("Divergence médiane   :", np.nanmedian(adiv))
print("(vitesses normalisées : vitesse max = 1)")

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
im = axs[0].imshow(np.log10(adiv + 1e-20))
plt.colorbar(im, ax=axs[0], shrink=0.8, label="log10 |div|")
axs[0].set_title("Divergence résiduelle (cellules intérieures)"); axs[0].axis('off')
axs[1].hist(np.log10(adiv[np.isfinite(adiv)] + 1e-20), bins=40)
axs[1].set_xlabel("log10 |div|"); axs[1].set_ylabel("nombre de cellules")
plt.savefig('figures/divergence.png', dpi=200, bbox_inches='tight')
plt.show()

dmax, pos = test_champ_constant(U, V, dom)
print("Test c = 1 : max |dc/dt| =", dmax, "en", pos)

Si la divergence max est de l'ordre de 1e-3, cela veut dire une source parasite de l'ordre de 0,1 % de la vitesse max par unité de longueur : à comparer à l'erreur de conservation observée sur la masse.

## 2. Influence du schéma sur les temps d'arrivée

On compare dans le labyrinthe l'upwind (ordre 1) et MUSCL (ordre 2, deux limiteurs) pour l'injection continue : mêmes vitesses, même CFL, seul le schéma change.

In [ ]:
schemas = {
    "upwind":         dict(schema='upwind', temps='euler'),
    "MUSCL minmod":   dict(schema='muscl', lim='minmod',  temps='rk2'),
    "MUSCL van Leer": dict(schema='muscl', lim='vanleer', temps='rk2'),
}
res = {nom: lancer(arret_sortie=0.99, **opt) for nom, opt in schemas.items()}

print(f"{'schéma':16s} {'t(10%)':>8} {'t(50%)':>8} {'t(90%)':>8} {'t(99%)':>8} {'c min':>10} {'c max':>8} {'temps calcul':>13}")
for nom, r in res.items():
    ts = [temps_arrivee(r['temps'], r['c_sortie'], s) for s in (0.1, 0.5, 0.9)]
    print(f"{nom:16s} {ts[0]:8.1f} {ts[1]:8.1f} {ts[2]:8.1f} {r['t_final']:8.1f} {r['cmin'].min():10.2e} {r['cmax'].max():8.4f} {r['duree']:11.2f} s")

plt.figure(figsize=(8, 4.5))
for nom, r in res.items():
    plt.plot(r['temps'], r['c_sortie'], label=nom)
plt.xlabel("t"); plt.ylabel("c à la sortie"); plt.title("Courbes de percée selon le schéma")
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('figures/percee_schemas.png', dpi=200, bbox_inches='tight')
plt.show()

**À interpréter** : si les courbes sont proches, le résultat est peu sensible à la diffusion numérique et l'étalement vient surtout de l'écoulement. Un front plus raide avec MUSCL indique que l'upwind diffuse davantage. Le temps de percée à 50 % est la grandeur la plus robuste.

## 3. Influence du pas de temps (CFL)

Pour un schéma stable, le résultat ne doit presque pas dépendre du CFL : on voit surtout l'erreur en temps de l'Euler explicite.

In [ ]:
print(f"{'CFL':>5} {'dt':>8} {'pas':>7} {'t(50%)':>9}")
for cfl_ in [0.25, 0.5, 0.9]:
    r = lancer(cfl=cfl_, arret_sortie=0.99)
    print(f"{cfl_:5.2f} {r['dt']:8.4f} {r['n']:7d} {temps_arrivee(r['temps'], r['c_sortie'], 0.5):9.1f}")

## 4. Carte des temps d'arrivée : où le traceur arrive-t-il, et quand ?

Pour chaque cellule, on enregistre le premier instant où c dépasse 0,5. Les cellules jamais atteintes (en blanc) sont les zones où le traceur n'arrive pas avant la fin du calcul : typiquement le fond des impasses.

In [ ]:
t_fin = 3.0 * res["upwind"]['t_final']
cartes = {}
for nom in ["upwind", "MUSCL van Leer"]:
    r = simuler(c0, U, V, t_end=t_fin, cfl=0.5, inlet=inlet, outlet=outlet, c_in=1.0, fluid=fluid,
                seuil_arrivee=0.5, **schemas[nom])
    cartes[nom] = r['t_arrivee']

fig, axs = plt.subplots(1, 3, figsize=(16, 5))
vmax_t = np.nanmax(cartes["upwind"])
for ax, (nom, tc) in zip(axs[:2], cartes.items()):
    im = ax.imshow(np.where(fluid, tc, np.nan), vmin=0, vmax=vmax_t)
    plt.colorbar(im, ax=ax, shrink=0.8, label="t d'arrivée (c ≥ 0.5)")
    ax.set_title(nom); ax.axis('off')
diff = cartes["MUSCL van Leer"] - cartes["upwind"]
lim_d = np.nanmax(np.abs(diff))
im = axs[2].imshow(np.where(fluid, diff, np.nan), cmap='RdBu_r', vmin=-lim_d, vmax=lim_d)
plt.colorbar(im, ax=axs[2], shrink=0.8, label="différence MUSCL - upwind")
axs[2].set_title("Écart entre schémas"); axs[2].axis('off')
plt.savefig('figures/carte_arrivee.png', dpi=200, bbox_inches='tight')
plt.show()

for nom, tc in cartes.items():
    print(f"{nom:16s} : cellules fluides jamais atteintes avant t = {t_fin:.0f} : {(fluid & np.isnan(tc)).sum()} sur {fluid.sum()}")

## 5. À propos de la résolution de l'image

L'énoncé demande de discuter l'influence de la résolution. Le potentiel φ vient d'un solveur fourni : si tu peux le relancer sur la même image à une résolution différente (par exemple deux fois plus de pixels), refais le notebook 1 puis les notebooks 6 et 7 sur ce nouveau jeu de données. Compare ensuite les temps d'arrivée **en unités physiques** (en tenant compte du fait que dx change, et que la normalisation des vitesses à 1 change l'unité de temps) et la divergence résiduelle.

## 6. Discussion à rédiger

Voici les points que l'énoncé attend, avec les chiffres à citer :

1. **Précision des vitesses** : divergence max et moyenne, où elle est maximale (coins ? bras étroits ?), effet sur la conservation de c.
2. **Stabilité** : CFL limite observée dans le notebook 3, pas de temps imposé dans le labyrinthe (quelle cellule le fixe ?).
3. **Schéma spatial** : diffusion numérique de l'upwind, oscillations des schémas non limités, ordre observé (notebook 4 et 5).
4. **Temps d'arrivée** : valeur à 50 %, dispersion (écart entre t(10 %) et t(90 %)), sensibilité au schéma et au CFL.
5. **Impasses et coins** : l'impasse se remplit-elle ? par quel mécanisme (il n'y a pas de diffusion physique : seul le transport par une vitesse non nulle, éventuellement très faible, y amène du traceur) ? Qu'en est-il dans les bras étroits où la vitesse est mal résolue (quelques cellules de large) ?
6. **Pourquoi la forme conservative** : un seul flux par face partagé entre deux cellules, donc masse conservée à la précision machine ; la forme non conservative crée une source ou un puits là où la divergence discrète n'est pas nulle ; elle traite mal les fronts raides.